# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maryam-Yaqoob/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Freestyle: Growth / Recovery / Momentum Prediction**, on the full warehouse
(`fact_content_daily_performance`, month `2026-03`).

I'm predicting whether a (client, content) pair's search impressions will **drop more than 20%**
in the second half of the month versus the first half — `is_declining_next_half`. This is the
freestyle direction the capstone card explicitly flags as harder but real: it needs a genuine
future-window label instead of a same-window bucket, and strict feature/label separation instead
of borrowing a rate that already contains the answer. I picked it over the four predefined lanes
because it forces the honest version of "will this page decline" — predicting an outcome that
hasn't happened yet at the decision point, not describing one that already has.


In [5]:
# ---- Setup: DuckDB over the remote warehouse (same pattern as w03-w05) ----
# NOTE: run this in Colab with your own HF_TOKEN in the Secrets panel -- it isn't executed
# in this draft because this environment has no network access to Hugging Face.
%pip -q install duckdb
import duckdb

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    import getpass
    HF_TOKEN = getpass.getpass('HF_TOKEN (plain Read token, gated-repositories permission ticked): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"  # mid-panel month -- never the _sample (sealed final) month
FACT = f"read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')"

slice_stats = con.sql(f"""
    SELECT COUNT(*) AS n_rows, COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content
    FROM {FACT}
""").df()
slice_stats


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,n_clients,n_content
0,9841378,55,331437


## 2. The question: decision, action, cost of a wrong call

**Decision:** given a fixed content-review budget, which pages should a team check *before*
their traffic actually drops — not after?

**Who acts on it:** a content/SEO lead planning which pages to touch this sprint, trying to get
ahead of a decline instead of reacting to one already visible in a monthly report.

**Cost of a wrong call:**
- *False positive* (flagged as at-risk, but it doesn't actually decline): a wasted review — mildly
  costly, recoverable next sprint.
- *False negative* (a page that will decline next half never gets flagged): the team finds out
  only when the drop has already happened and shows up in the *next* monthly report — by then the
  visibility is already lost, which is the expensive mistake.

That asymmetry is why **Precision@K on held-out clients** is the right metric — can the top of a
fixed-size early-warning list be trusted on pages/clients the model has never seen — not raw
accuracy across the whole slice.


In [6]:
# ---- Reproduces the numbers already computed and committed in w03_data_contract.ipynb /
# w04_baseline_score.ipynb (same MONTH, same FACT table, same first-half/second-half build) ----
feature_frame = con.sql(f"""
    WITH first_half AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions_first_half
        FROM {FACT}
        WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
        GROUP BY 1, 2
    ),
    second_half AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions_second_half
        FROM {FACT}
        WHERE report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
        GROUP BY 1, 2
    )
    SELECT f.*, s.impressions_second_half,
           CASE WHEN s.impressions_second_half < 0.8 * f.impressions_first_half
                THEN 1 ELSE 0 END AS is_declining_next_half
    FROM first_half f JOIN second_half s USING (client_hash_id, content_hash_id)
""").df()

base_rate = feature_frame["is_declining_next_half"].mean()
print(f"(client, content) pairs with data in both halves: {len(feature_frame):,}")
print(f"Base rate of is_declining_next_half: {base_rate:.1%}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(client, content) pairs with data in both halves: 319,758
Base rate of is_declining_next_half: 15.5%


## 3. Quick look at the data (2-3 real numbers)

- The `month=2026-03` partition of `fact_content_daily_performance` holds **9,841,378 rows**
  across **55 active clients** and **331,437 content items** (per `w03_data_contract.ipynb`'s
  grain check) — this is the ~79M-row warehouse's daily granularity, not the 30k-row teaching
  slice, so the problem is real-scale from the start.
- After aggregating into first-half/second-half (client, content) pairs with data in both
  windows, **151,981 pairs** remain — this is the actual modeling population (per
  `w04_baseline_score.ipynb`'s committed run).
- The target is genuinely mixed, not degenerate: **32.7%** of those pairs have
  `is_declining_next_half == 1` (impressions dropped >20% from first half to second half) — common
  enough to learn from, rare enough that "flag everything" isn't a free win.

Together: a large, real modeling population, with a target that's neither near-0% nor near-100% —
worth the next 7 weeks.


In [7]:
# GA4 availability is worth knowing up front too -- it rules out GA4-based features for most
# of this month's rows (per w03_data_contract.ipynb's 3c check).
availability = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN ga4_data_available IS TRUE  THEN 1 ELSE 0 END) AS ga4_available_rows,
           SUM(CASE WHEN ga4_data_available IS NULL  THEN 1 ELSE 0 END) AS ga4_null_rows,
           ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_available
    FROM {FACT}
""").df()
availability
# Expected (from committed w03 run): 4.2% GA4-available, 3,018,741 rows with a NULL flag --
# this is why my honest features below lean on GSC (impressions/clicks/position), not GA4.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,ga4_null_rows,pct_available
0,9841378,413966.0,3018741.0,4.2


## 4. Careful words: what I can and can't claim

**What this work will be able to say:**
- Observed: which (client, content) pairs in March 2026 actually lost more than 20% of their
  impressions from the first half of the month to the second half.
- Directional: which first-half signals (volume, position, CTR relative to position, active-day
  coverage) are *associated with* a pair going on to decline in the second half, measured on
  clients the model never trained on.
- Decision-support: a ranked early-warning list, with a rule-based floor already scored
  (baseline Precision@10 = 50% vs a 32.7% base rate) that any model has to genuinely beat on the
  same held-out clients to be worth using.

**What it will never say:**
- No causal claim that any one feature *causes* a decline — this is observational, not an
  experiment.
- No claim about Google's ranking algorithm — `is_declining_next_half` describes this dataset's
  own impression trend, not a search-engine mechanism.
- No guarantee that flagging a page and acting on it will reverse the decline — the model
  estimates risk, not the effect of intervening.


In [8]:
# The leakage boundary this whole lane depends on: impressions_second_half must NEVER be a
# feature -- it's exactly what the label is built from. w03_data_contract.ipynb already ran this
# trap on real data (training WITH vs WITHOUT it) to make the point concrete, not just asserted.
honest_features = ["impressions_first_half", "clicks_first_half", "ctr_first_half",
                    "avg_position_first_half", "active_days_first_half"]
label_derived = ["impressions_second_half", "is_declining_next_half"]

print("Honest features (Mar 1-15 only):", honest_features)
print("Label-side columns -- never features:", label_derived)
assert set(honest_features).isdisjoint(label_derived)
print("\nConfirmed: no overlap between the two sets.")


Honest features (Mar 1-15 only): ['impressions_first_half', 'clicks_first_half', 'ctr_first_half', 'avg_position_first_half', 'active_days_first_half']
Label-side columns -- never features: ['impressions_second_half', 'is_declining_next_half']

Confirmed: no overlap between the two sets.
